# ModelEdge — QLoRA Fine-tuning on Kaggle 2x T4

**Kaggle setup before running:**
1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **ON**
3. Settings → Secrets → add key `WANDB_API_KEY` with your W&B token
4. Run all cells top-to-bottom

Adapter weights are saved to `/kaggle/working/outputs/finetuned`.

In [ ]:
# Verify both GPUs are visible
!nvidia-smi

In [ ]:
# Install dependencies
# Install base stack first so unsloth can find them
!pip install -q peft accelerate datasets bitsandbytes wandb hf_transfer
# trl pinned to <2.0 — trl 2.x dropped SFTConfig compat used by Unsloth
!pip install -q 'trl>=1.6.0,<2.0.0'
# Unsloth + zoo installed last so they patch the already-installed trl/transformers
!pip install -q --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo
# Pin transformers — 5.0+ breaks Unsloth's compiled cache
!pip install -q 'transformers>=4.40.0,<5.0.0'

In [ ]:
# Verify imports
import unsloth
import torch
from unsloth import FastLanguageModel
print('torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
print('All imports OK')

In [ ]:
# Load W&B key from Kaggle Secrets
import os
from kaggle_secrets import UserSecretsClient
os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')

import wandb
wandb.login()

In [ ]:
# Clone repo and set up
%cd /kaggle/working
!rm -rf ModelEdge
!git clone -b dev/modeledge-pipeline https://github.com/sakshiasati17/ModelEdge.git ModelEdge
%cd /kaggle/working/ModelEdge
!git log --oneline -3

In [ ]:
# Prepare MedQA dataset
# Must use -m (module) to resolve relative imports inside data/
!PYTHONPATH=/kaggle/working/ModelEdge python -m data.prepare_dataset --dataset medqa --output data/processed/

In [ ]:
# Inspect a few training samples
import json

with open('data/processed/medqa/train.jsonl') as f:
    for i, line in enumerate(f):
        rec = json.loads(line)
        print(f'--- Sample {i+1} ---')
        print(rec['text'][:400])
        print()
        if i >= 2:
            break

In [ ]:
# Verify config output_dir and batch size
!grep -E 'output_dir|per_device_train_batch_size|max_seq_length' training/config.yaml

In [ ]:
# Always pull latest code before training
%cd /kaggle/working/ModelEdge
!git pull origin dev/modeledge-pipeline
!echo "--- trainer_utils packing setting ---"
!grep -E "packing|max_seq_length|processing_class" training/trainer_utils.py
!echo "--- deleting stale compiled cache ---"
!rm -rf /kaggle/working/ModelEdge/unsloth_compiled_cache
!echo "--- starting training ---"
!PYTHONPATH=/kaggle/working/ModelEdge PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m training.finetune --config training/config.yaml


In [ ]:
# Verify adapter files were saved
!ls -lh /kaggle/working/outputs/finetuned/

In [ ]:
# Quick inference test
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    '/kaggle/working/outputs/finetuned',
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

test_prompt = (
    'Below is a medical question. Answer it accurately and concisely.\n\n'
    '### Instruction:\nWhat is the first-line treatment for hypertension?\n\n'
    '### Input:\n\n'
    '### Response:\n'
)
inputs = tokenizer(test_prompt, return_tensors='pt').to('cuda')
out = model.generate(**inputs, max_new_tokens=128, do_sample=False)
print(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))

In [ ]:
# Serve the fine-tuned model with vLLM across both T4s
# Run in a Kaggle terminal tab (Add-ons → Terminal)
print('Open a terminal and run:')
print()
print('  python -m vllm.entrypoints.openai.api_server \\')
print('    --model /kaggle/working/outputs/finetuned \\')
print('    --tensor-parallel-size 2 \\')
print('    --host 0.0.0.0 --port 8001')